## 4.3.1 Kısa veri küpü
Olgu: bir pazarlama kaydı. Ölçüler: kayıt sayısı ve abonelik oranı. Boyutlar: yaş grubu, konut kredisi, eğitim. Yaş sınırları 30 ve 60 alan temelli sabittir; örneklem kantilleriyle optimize edilmez. Kişi kimliği olmadığı için kayıt sayısı eşsiz kişi sayısı değildir.



In [ ]:
kup=egitim.copy(); kup["abonelik"]=sonuc
kup["yas_grubu"]=pd.cut(kup.yas,[0,30,60,np.inf],right=False,labels=["30_alti","30_59","60_ustu"])
display(kup.groupby(["yas_grubu","konut_kredisi","egitim"],observed=True,dropna=False).abonelik.agg(["count","mean"]))
print("Roll-up: eğitim ve yaş toplanır")
display(kup.groupby("konut_kredisi").abonelik.agg(["count","mean"]))
print("Drill-down: yaş grubu açılır")
display(kup.pivot_table(index="yas_grubu",columns="konut_kredisi",values="abonelik",aggfunc=["count","mean"],observed=True))
genel=kup.groupby("konut_kredisi").abonelik.mean()
alt=kup.groupby(["yas_grubu","konut_kredisi"],observed=True).abonelik.mean().unstack()
fark=alt["yes"]-alt["no"]
print("Genel yes-no farkı",genel["yes"]-genel["no"],"alt grup farkları",fark.to_dict())
print("Tüm alt gruplarda yön ters mi?",bool((np.sign(fark)==-np.sign(genel["yes"]-genel["no"])).all()))


count      mean
yas_grubu konut_kredisi egitim                    
30_alti   no            primary       90  0.155556
                        secondary    729  0.146776
                        tertiary     363  0.173554
                        NaN           45  0.111111
          yes           primary      202  0.044554
                        secondary   1537  0.065712
                        tertiary     534  0.088015
                        NaN           60  0.066667
30_59     no            primary     2111  0.044529
                        secondary   6246  0.065482
                        tertiary    4581  0.097359
                        NaN          633  0.058452
          yes           primary     3010  0.047841
                        secondary  10034  0.050827
                        tertiary    4675  0.062032
                        NaN          628  0.046178
60_ustu   no            primary      160  0.212500
                        secondary    190  0.178947
                        tertiary     124  0.250000
                        NaN           39  0.153846
          yes           primary       54  0.092593
                        secondary     73  0.068493
                        tertiary      32  0.218750
                        NaN           18  0.111111

Roll-up: eğitim ve yaş toplanır


,count,mean
konut_kredisi,,
no,15311,0.083600
yes,20857,0.055281


Drill-down: yaş grubu açılır


count             mean          
konut_kredisi     no    yes        no       yes
yas_grubu                                      
30_alti         1227   2333  0.154034  0.069010
30_59          13571  18347  0.072655  0.053033
60_ustu          513    177  0.204678  0.107345

Genel yes-no farkı -0.028318825568839974 alt grup farkları {'30_alti': -0.08502437127762924, '30_59': -0.01962173250741061, '60_ustu': -0.09733372980473783}
Tüm alt gruplarda yön ters mi? False


**Küp ve Simpson sonucu:** Roll-up'ta konut kredisi olmayan 15.311 kaydın abonelik oranı %8,3600; kredisi olan 20.857 kaydın oranı %5,5281'dir. Kredi=yes eksi no farkı −2,8319 yüzde puandır. Yaş katmanlarında bu fark 30 altında −8,5024; 30–59'da −1,9622; 60 ve üzerinde −9,7334 yüzde puandır. Yön üç katmanda da negatiftir; bu karşılaştırmada Simpson tersine dönüşü bulunmamıştır. Paradoks üretmek için sınır veya grup seçimi değiştirilmez.

Farkın büyüklüğü yaşa göre değişir; özellikle 60 ve üzerindeki kredi=yes hücresi yalnız 177 kayıttır. Hücre oranları için güven aralığı veya fark testi üretilmediğinden nüfus düzeyinde kesinlik iddia edilmez. Olgu bir kayıt olduğundan bu sayılar eşsiz kişi sayısı değildir. Kredi etkisinin nedensel olduğu ya da başka boyutlarda da paradoks bulunmayacağı sonucuna varılmaz.

**Yorum:** Simpson denetimi yalnız bu konut kredisi/yaş karşılaştırmasına aittir; sonuç tüm değişkenlere genellenemez.

### Ek analiz · Veri küpünde %95 güven aralıkları

Mevcut oranlar ve hücreler aynen korunur. Her genel/yaş/eğitim hücresinde abonelik oranı için **Wilson**, kredi=yes eksi kredi=no farkı için iki bağımsız oranın Wilson sınırlarını birleştiren **Newcombe** yöntemi kullanılır. Basit Wald aralığı küçük gruplarda ve uç oranlarda zayıf davranabildiği için seçilmemiştir. Ayrı grup aralıklarının örtüşmesine bakmak yerine doğrudan **farkın aralığı** hesaplanır. Oran tabloları 0–1, fark tablosu yüzde puan birimindedir.

Bu hesaplar yeni model/CV deneyi değildir. Aynı kayıtlar, önceden sabit 30/60 yaş sınırları ve aynı gözlenen oranlar kullanılır. Yöntem kaynağı: [statsmodels oran farkı belgesi](https://www.statsmodels.org/dev/generated/statsmodels.stats.proportion.confint_proportions_2indep.html) ve [Wilson/Newcombe uygulaması](https://github.com/statsmodels/statsmodels/blob/main/statsmodels/stats/proportion.py). Ek bağımlılık gerektirmemek için formüller mevcut numpy/scipy ile yazılmıştır.


In [ ]:
# EK ANALIZ: Eski kup sonuclari degistirilmeden belirsizlik hesaplanir.
from scipy.stats import norm

def wilson_araligi(olumlu, kayit, alfa=0.05):
    olumlu=np.asarray(olumlu,dtype=float); kayit=np.asarray(kayit,dtype=float)
    if np.any(kayit<=0) or np.any(olumlu<0) or np.any(olumlu>kayit):
        raise ValueError("Gecerli olumlu sayisi ve pozitif kayit sayisi gerekli")
    z=norm.isf(alfa/2); oran=olumlu/kayit
    payda=1+z*z/kayit
    merkez=(oran+z*z/(2*kayit))/payda
    yari=z*np.sqrt(oran*(1-oran)/kayit+z*z/(4*kayit*kayit))/payda
    return np.clip(merkez-yari,0,1),np.clip(merkez+yari,0,1)

def oran_ozeti(tablo, boyutlar):
    ozet=tablo.groupby(boyutlar,observed=True,dropna=False).abonelik.agg(kayit="count",olumlu="sum",oran="mean")
    ozet["alt_95"],ozet["ust_95"]=wilson_araligi(ozet.olumlu,ozet.kayit)
    ozet["aralik_genisligi"]=ozet.ust_95-ozet.alt_95
    return ozet

def newcombe_farki(olumlu_bir,kayit_bir,olumlu_iki,kayit_iki):
    oran_bir=olumlu_bir/kayit_bir; oran_iki=olumlu_iki/kayit_iki
    alt_bir,ust_bir=wilson_araligi(olumlu_bir,kayit_bir)
    alt_iki,ust_iki=wilson_araligi(olumlu_iki,kayit_iki)
    fark=oran_bir-oran_iki
    return fark, fark-np.sqrt((oran_bir-alt_bir)**2+(ust_iki-oran_iki)**2), fark+np.sqrt((ust_bir-oran_bir)**2+(oran_iki-alt_iki)**2)

kup_genel_ga=oran_ozeti(kup,["konut_kredisi"])
kup_yas_ga=oran_ozeti(kup,["yas_grubu","konut_kredisi"])
kup_detay_ga=oran_ozeti(kup,["yas_grubu","konut_kredisi","egitim"])
print("Genel oranlar ve Wilson %95 araliklari (0-1)"); display(kup_genel_ga)
print("Yas katmanlari ve Wilson %95 araliklari (0-1)"); display(kup_yas_ga)
print("Uc boyutlu tum gozlenen hucreler (0-1)"); display(kup_detay_ga)
fark_satirlari=[]
for grup,tablo in [("tum_yaslar",kup)]+[(str(ad),alt_tablo) for ad,alt_tablo in kup.groupby("yas_grubu",observed=True)]:
    sayim=tablo.groupby("konut_kredisi").abonelik.agg(["sum","count"])
    fark,alt,ust=newcombe_farki(sayim.loc["yes","sum"],sayim.loc["yes","count"],sayim.loc["no","sum"],sayim.loc["no","count"])
    fark_satirlari.append({"yas_grubu":grup,"fark_yuzde_puan":100*fark,"alt_95_yuzde_puan":100*alt,"ust_95_yuzde_puan":100*ust,"sifiri_iceriyor":bool(alt<=0<=ust)})
kup_fark_ga=pd.DataFrame(fark_satirlari).set_index("yas_grubu")
print("Kredi yes eksi no: Newcombe %95 fark araliklari, yuzde puan"); display(kup_fark_ga)
for ad,satir in kup_fark_ga.iterrows():
    karar="sifiri icermiyor; bu varsayimlar altinda negatif farkla uyumlu" if satir.ust_95_yuzde_puan<0 else "sifiri iceriyor; farkin yonu bu aralikla kesinlestirilemiyor"
    display(Markdown(f"**Hesaplanan yorum — {ad}:** fark **{satir.fark_yuzde_puan:.2f} yüzde puan**, %95 aralık **[{satir.alt_95_yuzde_puan:.2f}; {satir.ust_95_yuzde_puan:.2f}]**; {karar}. Bu, kredi durumunun nedensel etkisi değildir."))
for ad,tablo in [("kup_genel_guven_araligi",kup_genel_ga),("kup_yas_guven_araligi",kup_yas_ga),("kup_detay_guven_araligi",kup_detay_ga),("kup_fark_guven_araligi",kup_fark_ga)]:
    tablo.to_csv(cikti/(ad+".csv"))


Genel oranlar ve Wilson %95 araliklari (0-1)
Yas katmanlari ve Wilson %95 araliklari (0-1)
Uc boyutlu tum gozlenen hucreler (0-1)
Kredi yes eksi no: Newcombe %95 fark araliklari, yuzde puan


,kayit,olumlu,oran,alt_95,ust_95,aralik_genisligi
konut_kredisi,,,,,,
no,15311,1280,0.083600,0.079320,0.088089,0.008770
yes,20857,1153,0.055281,0.052261,0.058465,0.006204


kayit  olumlu  ...    ust_95  aralik_genisligi
yas_grubu konut_kredisi                 ...                            
30_alti   no              1227     189  ...  0.175309          0.040391
          yes             2333     161  ...  0.080020          0.020603
30_59     no             13571     986  ...  0.077144          0.008736
          yes            18347     973  ...  0.056370          0.006487
60_ustu   no               513     105  ...  0.241726          0.069706
          yes              177      19  ...  0.161566          0.091761

[6 rows x 6 columns]

kayit  ...  aralik_genisligi
yas_grubu konut_kredisi egitim            ...                  
30_alti   no            primary       90  ...          0.149346
                        secondary    729  ...          0.051376
                        tertiary     363  ...          0.077812
                        NaN           45  ...          0.186586
          yes           primary      202  ...          0.058879
                        secondary   1537  ...          0.024838
                        tertiary     534  ...          0.048248
                        NaN           60  ...          0.133025
30_59     no            primary     2111  ...          0.017660
                        secondary   6246  ...          0.012277
                        tertiary    4581  ...          0.017175
                        NaN          633  ...          0.036828
          yes           primary     3010  ...          0.015283
                        secondary  10034  ...          0.008601
                        tertiary    4675  ...          0.013842
                        NaN          628  ...          0.033191
60_ustu   no            primary      160  ...          0.126000
                        secondary    190  ...          0.108668
                        tertiary     124  ...          0.150871
                        NaN           39  ...          0.224820
          yes           primary       54  ...          0.158898
                        secondary     73  ...          0.120912
                        tertiary      32  ...          0.277312
                        NaN           18  ...          0.296978

[24 rows x 6 columns]

,fark_yuzde_puan,alt_95_yuzde_puan,ust_95_yuzde_puan,sifiri_iceriyor
yas_grubu,,,,
tum_yaslar,-2.831883,-3.372962,-2.298394,False
30_alti,-8.502437,-10.836225,-6.296480,False
30_59,-1.962173,-2.510585,-1.422020,False
60_ustu,-9.733373,-15.007636,-3.403676,False


**Hesaplanan yorum — tum_yaslar:** fark **-2.83 yüzde puan**, %95 aralık **[-3.37; -2.30]**; sifiri icermiyor; bu varsayimlar altinda negatif farkla uyumlu. Bu, kredi durumunun nedensel etkisi değildir.

**Hesaplanan yorum — 30_alti:** fark **-8.50 yüzde puan**, %95 aralık **[-10.84; -6.30]**; sifiri icermiyor; bu varsayimlar altinda negatif farkla uyumlu. Bu, kredi durumunun nedensel etkisi değildir.

**Hesaplanan yorum — 30_59:** fark **-1.96 yüzde puan**, %95 aralık **[-2.51; -1.42]**; sifiri icermiyor; bu varsayimlar altinda negatif farkla uyumlu. Bu, kredi durumunun nedensel etkisi değildir.

**Hesaplanan yorum — 60_ustu:** fark **-9.73 yüzde puan**, %95 aralık **[-15.01; -3.40]**; sifiri icermiyor; bu varsayimlar altinda negatif farkla uyumlu. Bu, kredi durumunun nedensel etkisi değildir.

**Yorum ve karar:** Yukarıdaki hesaplanan yorumlar bu çalıştırmanın sayımlarından üretilir. Negatif kalan fark aralığı, bağımsız binom kayıt modeli altında kredi=yes grubunda daha düşük abonelikle uyumludur; kredinin bunu yarattığını göstermez. Simpson sonucu yine gözlenen yönlerin karşılaştırmasıdır; güven aralığı eklemek paradoksun tanımını değiştirmez. Küçük hücrelerdeki geniş aralıklar oranları kesin müşteri profilleri olarak sunmama gerekçesidir.

%95, aynı örnekleme süreci çok kez yinelendiğinde bu yöntemle kurulan aralıkların yaklaşık %95'inin ilgili oranı kapsaması anlamına gelmektedir; müşterinin abonelik olasılığının %95 olduğu anlamına gelmez. Bunlar noktasal, eşzamanlı çoklu karşılaştırmaya göre düzeltilmemiş aralıklardır. Aynı müşteri tekrarları ve zaman bağımlılığı denetlenemediğinden belirsizlik eksik tahmin edilebilir. Dönem değişimi, temsil yanlılığı ve gözlenmeyen karıştırıcılar aralıkların kapsamında değildir. Bu nedenle analiz gözlenen dönem için koşullu bir belirsizlik özeti olarak kullanılır.


In [ ]:
print("Slice: konut kredisi=yes")
display(kup.loc[kup.konut_kredisi.eq("yes")].groupby("yas_grubu",observed=True).abonelik.agg(["count","sum","mean"]))
print("Dice: 30 altı veya 60 üstü, konut kredisi=no")
alt_kup=kup.loc[kup.yas_grubu.isin(["30_alti","60_ustu"]) & kup.konut_kredisi.eq("no")]
display(alt_kup.groupby(["yas_grubu","egitim"],observed=True,dropna=False).abonelik.agg(["count","sum","mean"]))
kup_ozeti=kup.groupby(["yas_grubu","konut_kredisi","egitim"],observed=True,dropna=False).abonelik.agg(["count","sum","mean"])
agirlikli_oran=kup_ozeti["sum"].sum()/kup_ozeti["count"].sum()
assert kup_ozeti["count"].sum()==len(kup)
assert np.isclose(agirlikli_oran,kup.abonelik.mean())
print("Genel abonelik oranı",kup.abonelik.mean(),"özetlerden doğru roll-up",agirlikli_oran,"oranların basit ortalaması (yanlış)",kup_ozeti["mean"].mean())
kup_ozeti.to_csv(cikti/"kup_ozeti.csv")


Slice: konut kredisi=yes


,count,sum,mean
yas_grubu,,,
30_alti,2333,161,0.069010
30_59,18347,973,0.053033
60_ustu,177,19,0.107345


Dice: 30 altı veya 60 üstü, konut kredisi=no


count  sum      mean
yas_grubu egitim                         
30_alti   primary       90   14  0.155556
          secondary    729  107  0.146776
          tertiary     363   63  0.173554
          NaN           45    5  0.111111
60_ustu   primary      160   34  0.212500
          secondary    190   34  0.178947
          tertiary     124   31  0.250000
          NaN           39    6  0.153846

Genel abonelik oranı 0.06726940942269409 özetlerden doğru roll-up 0.06726940942269409 oranların basit ortalaması (yanlış) 0.1087868702948925


**Toplama kararının sayısal sonucu:** Hücrelerin pozitif toplamı/kayıt toplamı %6,7269 ile geliştirme oranını aynen verir. Hücre oranlarının basit ortalaması ise %10,8787'dir; yaklaşık 4,15 yüzde puanlık fazla sonuç, küçük hücrelere eşit ağırlık vermenin etkisidir. Bu nedenle roll-up'ta oranların ortalaması yerine sayım ve pozitif toplamı birleştirilir.

Slice'ta kredisi olan 60 ve üzeri grupta 19/177=%10,7345, 30 altında 161/2.333=%6,9010, 30–59'da 973/18.347=%5,3033 abonelik vardır. Dice'ta 60 ve üzeri, kredi=no, yüksek eğitim hücresi 31/124=%25 iken 30 altındaki karşılığı 63/363=%17,3554'tür. Bunlar seçilmiş kesitlerin betimlemeleridir; küçük hücrelerden genel hedefleme politikası çıkarılmaz. Eksik eğitim kategorisinin tutulması, özetlerin tüm kayıtları kapsamasını sağlar. `60_ustu` etiketi kodun [60,∞) aralığı nedeniyle 60 yaşını da kapsar.

**Yorum:** Yaş hiyerarşisi, ayrıntıdan gruba ve tüm yaşlara çıkmayı sağlar. Üç boyut, hiyerarşi düzeyleri ayrıca sayılmadan 2³=8 küboit üretir; tüm küboitleri saklamak bu küçük görev için gerekli değildir. Sayım ve pozitif toplamı genel oranı yeniden üretir; küçük hücrelerin oranlarına büyük hücrelerle eşit ağırlık vermek yanlış sonuca yol açabilir.

**Kapsam:** Üç boyutlu küpün roll-up/drill-down işlemleri özetleme düzeyini değiştirir; yeni bağımsız örneklem üretmez. Simpson kontrolü yalnız önceden seçilmiş yaş katmanlarındaki konut kredisi karşılaştırmasıdır. Daha ayrıntılı yaş/meslek/eğitim kontrolünün aynı sonucu vereceği sınanmamıştır.

**Yorum:** Simpson kontrolü, konut kredisi abonelik farkının yaş katmanlarında yön değiştirip değiştirmediğini sınar; tüm olası boyutlar taranmış değildir. Tam tersine dönüş bulunmazsa paradoks var denmez. Ağırlıklar ve grup boyutları oranlarla birlikte incelenir; ilişki nedensellik değildir.

## 4.3.2 Birliktelik · Seçenek B
Satırlar gerçek alışveriş sepeti değildir: her kayıt, öznitelik=değer öğeleriyle bir işlem kabul edilir. Sayısal yaş, bakiye, kampanya ve önceki temas sayısı için geliştirme verisinden üçte birlik sınırlar öğrenilir. Eşit genişlik uç bakiyeler yüzünden küçük gruplar yaratabilir; kantiller daha dengeli destek sağlar. Yinelenen kantiller birleştirilir; önceki temas sayısında sıfır yığılması tek aralık verince 0/pozitif ayrımı kullanılır. Gelecek küme sınır öğrenimine dahil değildir. Süre kullanılmaz. En çok üç öğeli kümeler (iki öncül + hedef) okunabilirlik/maliyet için seçilir. İlk keşifte destek .01 ve güven .20 yeterli pozitif kural üretmedi. Bu nedenle ay boyutu eklenerek destek .002 olarak revize edildi; .002/.005/.01 duyarlılığı gösterilir. Bu seçim veri görüldükten sonra yapılmıştır ve keşifseldir; bağımsız doğrulanmış bulgu değildir.

**Wu, Chen ve Han (2010), okuma notu:** Güven hedef yaygınlığına bağlıdır; lift taban oranına göre düzeltir ama nadir olaylarda büyüyebilir. Kulczynski iki koşullu olasılığın ortalamasıdır; lift ile aynı sıralamayı vermesi gerekmez. Bu nedenle tek ilgi ölçütüyle karar verilmez. Çok kural taraması sonrasında kurallar doğrulanmış nedensel bulgu sayılamaz.



In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
islem=egitim[["meslek","egitim","konut_kredisi","bireysel_kredi","iletisim","onceki_sonuc","ay"]].fillna("eksik").copy()
sinirlar={}
for sutun in ["yas","bakiye","kampanya","onceki_sayi"]:
    _, kesimler=pd.qcut(egitim[sutun],q=3,retbins=True,duplicates="drop")
    kesimler[0]=-np.inf; kesimler[-1]=np.inf
    if len(kesimler)==2 and sutun=="onceki_sayi": kesimler=np.array([-np.inf,0,np.inf])
    sinirlar[sutun]=kesimler.tolist()
    islem[sutun]=pd.cut(egitim[sutun],kesimler,include_lowest=True,labels=["dilim_"+str(i+1) for i in range(len(kesimler)-1)]).astype(str)
islem["hedef"]=sonuc.map({0:"no",1:"yes"})
sepet=pd.get_dummies(islem.astype(str),prefix_sep="=",dtype=bool)
zamanlar=[]
for destek in [.002,.005,.01]:
    basla=time.perf_counter(); kumeler=apriori(sepet,min_support=destek,use_colnames=True,max_len=3)
    zamanlar.append({"destek":destek,"saniye":time.perf_counter()-basla,"oruntu":len(kumeler)})
    if destek==.002: temel_kumeler=kumeler
kurallar=association_rules(temel_kumeler,metric="confidence",min_threshold=.20)
kurallar=kurallar[kurallar.consequents.apply(lambda deger:deger==frozenset(["hedef=yes"]))].copy()
kurallar["kulczynski"] = .5*(kurallar["confidence"]+kurallar["support"]/kurallar["consequent support"])
kurallar["dengesizlik"] = abs(kurallar["antecedent support"]-kurallar["consequent support"])/(kurallar["antecedent support"]+kurallar["consequent support"]-kurallar["support"])
display(pd.DataFrame(zamanlar)); print("Kantil sınırları",sinirlar)
for olcut in ["confidence","lift","kulczynski"]:
    print("İlk 10:",olcut); display(kurallar.sort_values(olcut,ascending=False)[["antecedents","support","confidence","lift","kulczynski","dengesizlik"]].head(10))
kurallar.to_csv(cikti/"hedef_kurallari.csv",index=False)
nadir_ogeler=set(sepet.columns[sepet.mean()<.05])
nadir=kurallar.loc[kurallar.antecedents.apply(lambda deger:bool(set(deger)&nadir_ogeler)).astype(bool)]
print("%5 altı öğe içeren kurallar"); display(nadir.sort_values("lift",ascending=False).head(10))


,destek,saniye,oruntu
0,0.002,1.171636,7582
1,0.005,0.853324,5766
2,0.010,0.755146,4229


Kantil sınırları {'yas': [-inf, 35.0, 45.0, inf], 'bakiye': [-inf, 148.0, 880.0, inf], 'kampanya': [-inf, 3.0, inf], 'onceki_sayi': [-inf, 0.0, inf]}
İlk 10: confidence


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,support,confidence,lift,kulczynski,dengesizlik
16773,"(ay=mar, kampanya=dilim_1)",0.002958,0.509524,7.574376,0.276751,0.876577
14324,"(iletisim=cellular, ay=mar)",0.003263,0.495798,7.370339,0.272149,0.859773
13172,"(bireysel_kredi=no, ay=mar)",0.003290,0.493776,7.340275,0.271343,0.857926
765,(ay=mar),0.003456,0.484496,7.202325,0.267937,0.847623
11348,"(ay=mar, konut_kredisi=no)",0.002544,0.484211,7.198079,0.261012,0.886211
16778,"(ay=mar, onceki_sayi=dilim_1)",0.003124,0.482906,7.178686,0.264675,0.861002
15747,"(ay=mar, onceki_sonuc=temas_yok)",0.003124,0.482906,7.178686,0.264675,0.861002
11176,"(ay=apr, konut_kredisi=no)",0.007576,0.390870,5.810519,0.251744,0.605594
661,(onceki_sonuc=success),0.002046,0.352381,5.238354,0.191398,0.865317
15544,"(onceki_sayi=dilim_2, onceki_sonuc=success)",0.002046,0.352381,5.238354,0.191398,0.865317


İlk 10: lift


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


... (4 outputs left out)

%5 altı öğe içeren kurallar


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,dengesizlik
16773,"(ay=mar, kampanya=dilim_1)",(hedef=yes),0.005806,0.067269,0.002958,0.509524,7.574376,1.0,0.002568,1.901684,0.873045,0.042192,0.474150,0.276751,0.876577
14324,"(iletisim=cellular, ay=mar)",(hedef=yes),0.006580,0.067269,0.003263,0.495798,7.370339,1.0,0.002820,1.849916,0.870046,0.046220,0.459435,0.272149,0.859773
13172,"(bireysel_kredi=no, ay=mar)",(hedef=yes),0.006663,0.067269,0.003290,0.493776,7.340275,1.0,0.002842,1.842525,0.869559,0.046575,0.457267,0.271343,0.857926
765,(ay=mar),(hedef=yes),0.007133,0.067269,0.003456,0.484496,7.202325,1.0,0.002976,1.809357,0.867343,0.048714,0.447317,0.267937,0.847623
11348,"(ay=mar, konut_kredisi=no)",(hedef=yes),0.005253,0.067269,0.002544,0.484211,7.198079,1.0,0.002190,1.808355,0.865621,0.036349,0.447011,0.261012,0.886211
16778,"(ay=mar, onceki_sayi=dilim_1)",(hedef=yes),0.006470,0.067269,0.003124,0.482906,7.178686,1.0,0.002689,1.803793,0.866304,0.044244,0.445613,0.264675,0.861002
15747,"(ay=mar, onceki_sonuc=temas_yok)",(hedef=yes),0.006470,0.067269,0.003124,0.482906,7.178686,1.0,0.002689,1.803793,0.866304,0.044244,0.445613,0.264675,0.861002
661,(onceki_sonuc=success),(hedef=yes),0.005806,0.067269,0.002046,0.352381,5.238354,1.0,0.001655,1.440246,0.813826,0.028805,0.305674,0.191398,0.865317
15544,"(onceki_sayi=dilim_2, onceki_sonuc=success)",(hedef=yes),0.005806,0.067269,0.002046,0.352381,5.238354,1.0,0.001655,1.440246,0.813826,0.028805,0.305674,0.191398,0.865317


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Apriori sonuçları ve eşik kararı:** Destek 0,002 / 0,005 / 0,01 için sırasıyla 7.582 / 5.766 / 4.229 sık öğe kümesi ve 1,171636 / 0,853324 / 0,755146 saniye görülür. Bunlar kural sayıları değil, en fazla üç öğeli sık küme sayılarıdır. Düşük destek bu çalıştırmada daha çok örüntü ve süre üretmiştir; süre tek ölçüm olduğundan hız üstünlüğü veya ortalama ± std diye sunulmaz. Seçenek B kullanıldığından FP-Growth ve 200 ilgisiz öğe ekleme deneyi zorunlu A seçeneği görevleriyle karıştırılmaz.

0,002 destek 36.168 kayıtta en az 73 ortak kayıt gerektirir. Eşik keşif sırasında düşürüldüğü için bulunan kurallar keşifseldir. Kantil sınırları yaşta 35/45, bakiyede 148/880 avrodur; kampanyada tekrar eden kantiller nedeniyle iki dilim, önceki sayıda 0/pozitif ayrımı kalmıştır. Üç dilim hedefi her değişkende gerçekleşmiş gibi yazılmaz. `pd.cut` sağdan kapalı olduğundan kampanya dilim_1 en fazla 3 teması, yaşın ilk dilimi 35 ve altını içerir. Bunlar küpteki [0,30), [30,60), [60,∞) sınırlarıyla aynı değildir.

En üst kural {Mart, kampanya≤3} için destek 0,002958, güven 0,509524, lift 7,574376, Kulczynski 0,276751 ve IR 0,876577'dir. Martın kendi başına güveni 0,484496 ve lift'i 7,202325'tir; daha ayrıntılı kuralın yükselişi bağımsız doğrulanmış ek katkı sayılmaz. Tüm sonuçlar `hedef=yes` olduğu için lift=güven/0,067269 olur; bu tabloda güven ve lift sıralamalarının aynı olması beklenir. Kulczynski kapsama yönünü de içerir: {Mart, önceki sayı=0} 0,264675 ile {Mart, kredi=no} 0,261012'nin önüne geçerken lift sırası tersidir. Bu, ölçütlerin neden birlikte okunması gerektiğinin somut örneğidir.

Nadir `onceki_sonuc=success` kuralının desteği 0,002046, güveni 0,352381, lift'i 5,238354 iken Kulczynski 0,191398 ve IR 0,865317'dir. Yüksek lift, küçük ve dengesiz bir öncülün geniş müşteri kitlesini kapsadığı anlamına gelmez. Kararımız destek, güven, lift, Kulczynski ve grup büyüklüğünü birlikte raporlamak; yüksek lift'i tek başına uygulanabilirlik gerekçesi yapmamaktır.

**Yorum:** En yüksek lift ve Kulczynski farklı öncül taban oranlarını farklı ödüllendirir. Destek .002, yaklaşık en az geliştirme satırlarının %0.2'si kadar ortak kayıt gerektirir; en nadir kategoriler için kural bulunmaması beklenir. Daha düşük destek daha çok ve daha oynak kural üretir. Yukarıdaki süre ölçümleri bu makineye aittir; algoritma hızının genellenebilir kanıtı değildir. Seçenek B için Apriori/FP-Growth karşılaştırması zorunlu değildir.

Aşağıdaki hücre en güçlü üç kuralı sayılarıyla yorumlar; model önemiyle örtüşme hedef sızıntısı olmayan geliştirmede eğitilmiş sabit modelin ileri dönem permütasyon önemiyle betimsel incelenir.



### Ek analiz · Birliktelik kurallarının güven değerine %95 aralık

Kural güveni P(abonelik=yes | öncül), öncülü sağlayan kayıtlar içindeki abonelik oranıdır. Her mevcut kural için öncül ve ortak sayıları yeniden sayılır, eski destek/güven ile eşitliği denetlenir ve güven için **Wilson %95 aralığı** hesaplanır. Payda tüm veri değil, öncülü sağlayan kayıtların sayısıdır. Önceki destek/güven/lift/Kulczynski değerleri ve kural seçimi korunur; aralık eklemek yeni kurallar keşfetmez ve mevcut sıralamayı yeniden optimize etmez.


In [ ]:
# EK ANALIZ: Her mevcut kuralin guveni icin Wilson araligi.
# Hicbir destek, guven esigi, kural sirasi veya eski sonuc degistirilmez.
kural_ga_satirlari=[]
for _,satir in kurallar.iterrows():
    maske=np.ones(len(islem),dtype=bool)
    for oge in satir.antecedents:
        sutun,deger=oge.split("=",1)
        maske &= islem[sutun].astype(str).eq(deger).to_numpy()
    oncul_sayisi=int(maske.sum())
    ortak_sayisi=int((maske & islem.hedef.eq("yes").to_numpy()).sum())
    guven=ortak_sayisi/oncul_sayisi
    assert np.isclose(guven,satir["confidence"])
    assert np.isclose(ortak_sayisi/len(islem),satir["support"])
    alt,ust=wilson_araligi(ortak_sayisi,oncul_sayisi)
    kural_ga_satirlari.append({"kural":" & ".join(sorted(satir.antecedents))+" -> hedef=yes", "oncul_sayisi":oncul_sayisi,"ortak_sayisi":ortak_sayisi,"destek":satir["support"],"guven":guven,"guven_alt_95":float(alt),"guven_ust_95":float(ust),"lift":satir["lift"],"kulczynski":satir["kulczynski"]})
kural_guven_ga=pd.DataFrame(kural_ga_satirlari).sort_values("kulczynski",ascending=False)
print("Mevcut pozitif kurallarin tamami; guven ve araliklar 0-1 olceginde")
display(kural_guven_ga.reset_index(drop=True))
for _,satir in kural_guven_ga.head(3).iterrows():
    display(Markdown(f"**Hesaplanan yorum — {satir.kural}:** {satir.ortak_sayisi}/{satir.oncul_sayisi} kayıtta abonelik; güven **%{100*satir.guven:.2f}**, noktasal %95 Wilson aralığı **[%{100*satir.guven_alt_95:.2f}; %{100*satir.guven_ust_95:.2f}]**. Bu aralık, sabit bir profil ve bağımsız kayıtlar varsayımına dayanır; çok sayıda adaydan kural seçmenin iyimserliğini veya gelecek dönemdeki değişimi kapsamaz."))
kural_guven_ga.to_csv(cikti/"kural_guven_araliklari.csv",index=False)


Mevcut pozitif kurallarin tamami; guven ve araliklar 0-1 olceginde


,kural,oncul_sayisi,ortak_sayisi,destek,guven,guven_alt_95,guven_ust_95,lift,kulczynski
0,ay=mar & kampanya=dilim_1 -> hedef=yes,210,107,0.002958,0.509524,0.442350,0.576356,7.574376,0.276751
1,ay=mar & iletisim=cellular -> hedef=yes,238,118,0.003263,0.495798,0.432851,0.558879,7.370339,0.272149
2,ay=mar & bireysel_kredi=no -> hedef=yes,241,119,0.003290,0.493776,0.431249,0.556498,7.340275,0.271343
3,ay=mar -> hedef=yes,258,125,0.003456,0.484496,0.424190,0.545257,7.202325,0.267937
4,ay=mar & onceki_sonuc=temas_yok -> hedef=yes,234,113,0.003124,0.482906,0.419675,0.546689,7.178686,0.264675
5,ay=mar & onceki_sayi=dilim_1 -> hedef=yes,234,113,0.003124,0.482906,0.419675,0.546689,7.178686,0.264675
6,ay=mar & konut_kredisi=no -> hedef=yes,190,92,0.002544,0.484211,0.414170,0.554876,7.198079,0.261012
7,ay=apr & konut_kredisi=no -> hedef=yes,701,274,0.007576,0.390870,0.355438,0.427492,5.810519,0.251744
8,onceki_sonuc=success -> hedef=yes,210,74,0.002046,0.352381,0.290950,0.419115,5.238354,0.191398
9,onceki_sayi=dilim_2 & onceki_sonuc=success -> ...,210,74,0.002046,0.352381,0.290950,0.419115,5.238354,0.191398


**Hesaplanan yorum — ay=mar & kampanya=dilim_1 -> hedef=yes:** 107/210 kayıtta abonelik; güven **%50.95**, noktasal %95 Wilson aralığı **[%44.23; %57.64]**. Bu aralık, sabit bir profil ve bağımsız kayıtlar varsayımına dayanır; çok sayıda adaydan kural seçmenin iyimserliğini veya gelecek dönemdeki değişimi kapsamaz.

**Hesaplanan yorum — ay=mar & iletisim=cellular -> hedef=yes:** 118/238 kayıtta abonelik; güven **%49.58**, noktasal %95 Wilson aralığı **[%43.29; %55.89]**. Bu aralık, sabit bir profil ve bağımsız kayıtlar varsayımına dayanır; çok sayıda adaydan kural seçmenin iyimserliğini veya gelecek dönemdeki değişimi kapsamaz.

**Hesaplanan yorum — ay=mar & bireysel_kredi=no -> hedef=yes:** 119/241 kayıtta abonelik; güven **%49.38**, noktasal %95 Wilson aralığı **[%43.12; %55.65]**. Bu aralık, sabit bir profil ve bağımsız kayıtlar varsayımına dayanır; çok sayıda adaydan kural seçmenin iyimserliğini veya gelecek dönemdeki değişimi kapsamaz.

**Yorum ve karar:** Yaklaşık %51 güvene sahip bir profilin aralığı, o oranın nokta tahmininden daha belirsiz olduğunu görünür kılar. Bu, “aynı profile arama yapınca kesin %51 başarı elde edilir” yorumunu desteklemez. İlk üç kural aynı Mart kayıtlarını paylaşabileceğinden aralıklar da bağımsız üç kanıt değildir. Ayrı aralıkların örtüşmesi kuralların eşitliğini veya farklılığını sınayan bir test sayılmaz.

Buradaki Wilson aralıkları **öncül sabit kabul edilerek hesaplanan keşifsel, noktasal aralıklardır**. Kurallar çok sayıda adaydan seçilmiş, destek eşiği de keşif sırasında değiştirilmiştir; nominal %95 kapsama seçilmiş kurallar için garanti edilmez. Genel taban oranıyla iç içe kayıtlar kullanıldığı için kural aralığını genel oranın aralığıyla karşılaştırıp bağımsız iki grup testi yapılmaz. Lift ve Kulczynski için bu hücrede güven aralığı hesaplanmamıştır; güven sınırını taban oranına bölmek lift'in bütün belirsizliğini kapsamaz.

**Bu çalışmanın kararı:** Bu profiller yeni veriyle doğrulanacak aday ilişkilerdir. Güven aralığı eklemek örneklem büyüklüğünü görünür kılar; geleceğe genellemeyi doğrulamaz. Doğrulama için öncüller ve eşikler sabitlenerek yeni, henüz incelenmemiş dönemde destek/güven tekrar ölçülmelidir. Mevcut ileri dönem daha önce incelendiği için burada yeni dokunulmamış doğrulama diye kullanılmamıştır.


In [ ]:
from sklearn.inspection import permutation_importance
onem=permutation_importance(en_iyi,gelecek,gelecek_sonuc,scoring="average_precision",n_repeats=5,random_state=TOHUM,n_jobs=1)
onemler=pd.Series(onem.importances_mean,index=egitim.columns).sort_values(ascending=False)
onem_tablosu=pd.DataFrame({"AP_azalmasi_ortalama":onem.importances_mean,"AP_azalmasi_std":onem.importances_std},index=gelecek.columns).sort_values("AP_azalmasi_ortalama",ascending=False)
display(onem_tablosu.head(10))
onem_tablosu.to_csv(cikti/"ileri_donem_permutasyon_onemi.csv")
for _,satir in kurallar.sort_values("kulczynski",ascending=False).head(3).iterrows():
    ogeler=sorted(satir.antecedents)
    display(Markdown(f"**Yorum:** {ogeler} koşullarında abonelik oranı %{100*satir['confidence']:.1f}; taban oranına göre lift {satir['lift']:.2f}, Kulczynski {satir['kulczynski']:.3f}. Bu profil, gözlenen kampanyadaki ilişkiyi gösterir; bu gruba arama yapmanın ek etkisini ölçmez. Öncül sütunlarının modelin ilk beş önemiyle kesişimi: {set(o.split('=')[0] for o in ogeler)&set(onemler.head(5).index)}."))

print("Alan yorumu: Mart/Nisan kurallari mevsim, kampanya politikasi ve musteri bilesimini birlikte yansitabilir; takvim nedensel etki degildir.")
print("Alan yorumu: onceki_sonuc=success daha onceki olumlu temas gecmisini gosterir; yeni musteriye ayni duzeyde genellenemez.")
print("Alan yorumu: konut kredisi/meslek/egitim profilleri gozlemseldir; kredi veya egitim durumunu degistirmenin aboneligi degistirecegi sonucu cikarilamaz.")


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,AP_azalmasi_ortalama,AP_azalmasi_std
onceki_sonuc,0.083016,0.001387
ay,0.060455,0.005881
yas,0.011325,0.002807
gun,0.006221,0.000730
bakiye,0.005691,0.000589
konut_kredisi,0.003167,0.000496
medeni_durum,0.003081,0.001554
egitim,0.002934,0.000666
kampanya,0.001029,0.000427
onceki_gun,0.000933,0.000365


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Yorum:** ['ay=mar', 'kampanya=dilim_1'] koşullarında abonelik oranı %51.0; taban oranına göre lift 7.57, Kulczynski 0.277. Bu profil, gözlenen kampanyadaki ilişkiyi gösterir; bu gruba arama yapmanın ek etkisini ölçmez. Öncül sütunlarının modelin ilk beş önemiyle kesişimi: {'ay'}.

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Yorum:** ['ay=mar', 'iletisim=cellular'] koşullarında abonelik oranı %49.6; taban oranına göre lift 7.37, Kulczynski 0.272. Bu profil, gözlenen kampanyadaki ilişkiyi gösterir; bu gruba arama yapmanın ek etkisini ölçmez. Öncül sütunlarının modelin ilk beş önemiyle kesişimi: {'ay'}.

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Yorum:** ['ay=mar', 'bireysel_kredi=no'] koşullarında abonelik oranı %49.4; taban oranına göre lift 7.34, Kulczynski 0.271. Bu profil, gözlenen kampanyadaki ilişkiyi gösterir; bu gruba arama yapmanın ek etkisini ölçmez. Öncül sütunlarının modelin ilk beş önemiyle kesişimi: {'ay'}.

Alan yorumu: Mart/Nisan kurallari mevsim, kampanya politikasi ve musteri bilesimini birlikte yansitabilir; takvim nedensel etki degildir.
Alan yorumu: onceki_sonuc=success daha onceki olumlu temas gecmisini gosterir; yeni musteriye ayni duzeyde genellenemez.
Alan yorumu: konut kredisi/meslek/egitim profilleri gozlemseldir; kredi veya egitim durumunu degistirmenin aboneligi degistirecegi sonucu cikarilamaz.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Üç kuralın alan yorumu:** {Mart, kampanya≤3} kuralı 210 öncül kaydın 107'sinde abonelik (%50,95); {Mart, cellular} 238 kaydın 118'inde (%49,58); {Mart, bireysel kredi=no} 241 kaydın 119'unda (%49,38) gözlenmiştir. Ortak destekleri sırasıyla yaklaşık %0,2958, %0,3263 ve %0,3290'dır. Bu üç profil büyük ölçüde aynı Mart kayıtlarını paylaşabilir; üç bağımsız doğrulama sayılmaz. Az temasla görülen yüksek oran “temas sayısını azaltmak aboneliği artırır”, cep telefonu koşulu “kanalı değiştirmek aynı sonucu verir” veya kredi=no koşulu “krediyi kapatmak aboneliği artırır” anlamına gelmez. Kampanya politikası, seçilmiş müşteri bileşimi ve dönem etkileri olası ortak açıklamalardır.

İleri dönem permütasyonunda AP azalması `onceki_sonuc` için 0,083016 ± 0,001387; `ay` için 0,060455 ± 0,005881; `yas` için 0,011325 ± 0,002807; `gun` için 0,006221 ± 0,000730; `bakiye` için 0,005691 ± 0,000589'dur. İlk üç kuralın ilk beş önemli sütunla kesişimi yalnız `ay`dır. Bu örtüşme kısmi ve betimseldir; diğer öncüllerin modelde önemsiz ya da kuralların yanlış olduğunu kanıtlamaz. Önceki başarı kuralı da modelin birinci sıradaki geçmiş sonuç bilgisiyle uyumludur. Kurallar geliştirmeden, önem ileri dönemden geldiği için sayısal büyüklükleri aynı etki ölçüsü gibi kıyaslanmaz.

Buradaki ± değerleri beş permütasyonun oynaklığıdır; model yeniden eğitimi, müşteri örneklemesi veya dönem belirsizliğini kapsamaz. İlişkili değişkenler birbirinin önemini maskeleyebilir. Gelecekteki uygulama için ayrı veriyle destek ve güvenin kararlılığı denetlenmelidir; burada böyle bir doğrulama yapılmış değildir.

**Yorum:** İleri dönem önemleri ve kurallar arasındaki örtüşme keşifseldir; önem nedensel etki sayılmaz.

## WEKA · bağımsız kontrol ve tekrarlama
Sınıflandırma için aynı geliştirme tablosu (süre yok, hedef son nominal sütun) ARFF yazılır. Eksikler `?` olur. ZeroR ve J48 (`-C 0.25 -M 30`) 10 kat CV, seed 42 ile çalışır. J48 C4.5'tir; Python CART ile aynı algoritma değildir. WEKA/Python rastgele bölmeleri de seed aynı olsa bile aynı olmayabilir. Bağımsız kontrol, büyüklük ve baseline tutarlılığıdır; satır satır skor eşitliği beklenmez. WEKA çıktısındaki kat ağırlıklı F1 ile Python pozitif F1 karıştırılmaz.

Birliktelikte ayrıklaştırılmış aynı tablo kullanılır: Apriori `-A -c last -T 0 -C 0.2 -M destek -U 1 -N 2147483647`. CAR modunda N=2147483647 tüm kuralları istenen alt destekten çıkarır; U=1 üst destek elemesini önler. WEKA desteği sayıya yuvarladığından alt eşik ceil(.002*N)/N ile tam 73 ortak kayda eşitlenir. WEKA antecedent uzunluğu sınırını Python gibi kısıtlamaz; karşılaştırmada en çok iki öncüllü ve `hedef=yes` kuralları filtrelenir. GUI: Explorer → Preprocess → ARFF aç; Classify → ZeroR/J48 → 10-fold/seed42; Associate → Apriori → yukarıdaki seçenekler. Çıktıları saklayın.



In [ ]:
def arff_yaz(tablo,yol):
    import csv
    def tirnak(deger):
        return "'"+str(deger).replace("\\","\\\\").replace("'","\\'")+"'"
    satirlar=["@relation banka"]
    for sutun in tablo:
        if pd.api.types.is_numeric_dtype(tablo[sutun]): tur="numeric"
        else: tur="{"+",".join(tirnak(deger) for deger in sorted(tablo[sutun].dropna().unique()))+"}"
        satirlar.append("@attribute "+tirnak(sutun)+" "+tur)
    satirlar.append("@data")
    for satir in tablo.itertuples(index=False,name=None):
        satirlar.append(",".join("?" if pd.isna(deger) else str(deger) if isinstance(deger,(int,float,np.number)) else tirnak(deger) for deger in satir))
    Path(yol).write_text("\n".join(satirlar),encoding="utf-8")
weka_veri=egitim.copy(); weka_veri["hedef"]=sonuc.map({0:"no",1:"yes"})
arff_yaz(weka_veri,cikti/"weka_siniflandirma.arff")
arff_yaz(islem,cikti/"weka_birliktelik.arff")
import shutil, urllib.request, os
java_yolu=shutil.which("java")
yerel_java=Path("/Applications/weka-3.8.7.app/Contents/runtime/Contents/Home/bin/java")
yerel_jar=Path("/Applications/weka-3.8.7.app/Contents/app/weka.jar")
if yerel_java.exists(): java_yolu=str(yerel_java)
weka_jar=yerel_jar if yerel_jar.exists() else cikti/"weka.jar"
if not weka_jar.exists():
    urllib.request.urlretrieve("https://repo.maven.apache.org/maven2/nz/ac/waikato/cms/weka/weka-stable/3.8.6/weka-stable-3.8.6.jar",weka_jar)
# Maven dağıtımında gerekli çekirdek bağımlılıkları da indirilir.
weka_sinif_yolu=str(weka_jar)
if not yerel_jar.exists():
    bagimliliklar=[("bounce.jar","nz/ac/waikato/cms/weka/thirdparty/bounce/0.18/bounce-0.18.jar"),("cup.jar","com/github/vbmacher/java-cup-runtime/11b-20160615/java-cup-runtime-11b-20160615.jar")]
    for ad,adres in bagimliliklar:
        yol=cikti/ad
        if not yol.exists(): urllib.request.urlretrieve("https://repo.maven.apache.org/maven2/"+adres,yol)
    weka_sinif_yolu=os.pathsep.join([str(weka_jar)]+[str(cikti/ad) for ad,_ in bagimliliklar])
if java_yolu is None: raise RuntimeError("Java bulunamadı. Colab Java çalışma zamanını veya Java 17+ kurulumunu kullanın.")
for ad,secenek in [("ZeroR",["weka.classifiers.rules.ZeroR"]),("J48",["weka.classifiers.trees.J48","-C","0.25","-M","30"])]:
    komut=[java_yolu,"--add-opens","java.base/java.lang=ALL-UNNAMED","-cp",weka_sinif_yolu]+secenek+["-t",str(cikti/"weka_siniflandirma.arff"),"-x","10","-s","42"]
    islem_sonucu=subprocess.run(komut,capture_output=True,text=True)
    if islem_sonucu.returncode or "Exception" in islem_sonucu.stderr or "exception:" in islem_sonucu.stderr: raise RuntimeError(islem_sonucu.stderr)
    (cikti/("weka_"+ad+".txt")).write_text(islem_sonucu.stdout)
    print(ad,islem_sonucu.stdout[-4500:])
komut=[java_yolu,"--add-opens","java.base/java.lang=ALL-UNNAMED","-Xmx2g","-cp",weka_sinif_yolu,"weka.associations.Apriori","-t",str(cikti/"weka_birliktelik.arff"),"-A","-c","last","-T","0","-C","0.2","-M",str(np.ceil(.002*len(islem))/len(islem)),"-U","1","-N","2147483647"]
islem_sonucu=subprocess.run(komut,capture_output=True,text=True)
if islem_sonucu.returncode or "Exception" in islem_sonucu.stderr or "exception:" in islem_sonucu.stderr: raise RuntimeError(islem_sonucu.stderr)
(cikti/"weka_Apriori.txt").write_text(islem_sonucu.stdout)
print(islem_sonucu.stdout[:1500])


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ZeroR [0.023s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.023s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

=== Classifier model (full training set) ===

ZeroR predicts class value: no

Time taken to build model: 0.17 seconds

Time taken to test model on training data: 0.7 seconds

=== Error on training data ===

Correctly Classified Instances       33735               93.2731 %
Incorrectly Classified Instances      2433                6.7269 %
Kappa statistic                          0     
Mean absolute error                      0.1255
Root mean squared error                  0.2505
Relative absolute error                100      %
Root relative squared error            100      %
Total Number of Instances            36168     


=== Detailed Accuracy By Clas

Görüntülenen çıkış son 5000 satıra kısaltıldı.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tz

J48 [0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Options: -C 0.25 -M 30 

=== Classifier model (full training set) ===

J48 pruned tree
------------------
: no (36168.0/2433.0)

Number of Leaves  : 	1

Size of the tree : 	1


Time taken to build model: 7.49 seconds

Time taken to test model on training data: 0.87 seconds

=== Error on training data ===

Correctly Classified Instances       33735               93.2731 %
Incorrectly Classified Instances      2433                6.7269 %
Kappa statistic                          0     
Mean absolute error                      0.1255
Root mean squared error                  0.2505
Relative absolute error                 99.9835 %
Root relative squared error    

Görüntülenen çıkış son 5000 satıra kısaltıldı.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tz

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Apriori

Minimum support: 0 (73 instances)
Minimum metric <confidence>: 0.2
Number of cycles performed: 1

Generated sets of large itemsets:

Size of set of large itemsets L(1): 87

Size of set of large itemsets L(2): 1126

Size of set of large itemsets L(3): 6769

Size of set of large itemsets L(4): 21629

Size of set of large itemsets L(5): 40088

Size of set of large itemsets L(6): 45698

Size of set of large itemsets L(7): 32607

Size of set of large itemsets L(8): 14234

Size of set of large itemsets L(9): 3574

Size of set of large itemsets L(10): 456

Size of set of large itemsets L(11): 22

Best rules found:

         1. onceki_sonuc=failure ay=jan 245 =

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**WEKA sınıflandırmasının gerçek çıktısı:** ZeroR CV'de %93,2731 doğruluk alırken tüm 2.433 olumlu kaydı kaçırır; ROC alanı 0,500'dür. Bu, çoğunluk doğruluğunu başarı saymama kararını destekler. J48'in tam geliştirmede eğitilen ağacı tek yapraklıdır; ancak CV sonucu tamamen aynı değildir: TN=33.730, FP=5, FN=2.426, TP=7. Yalnız 7/2.433 olumlu kayıt yakalanır (yaklaşık %0,288), pozitif F1 yaklaşık 0,006, ROC alanı 0,503, PRC alanı 0,070'tir. %93,2786 doğruluk ZeroR'dan çok az yüksekken pozitif yakalama hâlâ çok düşüktür. Eğitimdeki tek yaprak sonucunu bütün CV katları için aynen geçerli saymıyoruz.

WEKA PRC alanı, sklearn average_precision ile aynı sayısal tanıma sahip kabul edilmez; özellikle Weighted Avg. PRC/F1 pozitif sınıf ölçütü değildir. J48 ile Python CART'ın farkını yalnız uygulama hatası veya yalnız budama diye açıklamak yerine algoritma, ağırlık ve kat üyeliği farklarını birlikte değerlendiriyoruz. Java logları kayıtlı gerçek çalıştırmayı belgeliyor; yorum düzenlemesinde yeni Java sonucu üretilmedi.

**Yorum:** Bu hücre Java ile WEKA’yı gerçekten çalıştırır. Aynı tohum/kat sayısı Python ile aynı kat üyeliğini sağlamaz; loglar yeniden çalıştırmanın kanıtıdır.

**Apriori kontrolünün kapsamı:** WEKA'nın ilk metin parçasında görülen `hedef=no` ve uzun öncüllü kurallar, Python'daki kısa pozitif kuralların yokluğu anlamına gelmez. Kontrol hücresi tam logu okuyarak yalnız `hedef=yes` ve bir–iki öncül koşulunu filtreler. WEKA'daki “Minimum support: 0 (73 instances)” yazımı desteğin sıfır seçildiği şeklinde yorumlanmaz; sayısal eşik 73 ortak kayda eşlenmiştir. Aynı veri, ayrıklaştırma ve eşiklerle tamlık ve sayımlar aşağıdaki assert kontrolleriyle sınanır; ilk birkaç kuralın benzemesiyle yetinilmez.

**Yorum:** WEKA çıktıları gerçek Java çalıştırmasına aittir. ZeroR çoğunluk sınıfını seçer; pozitif yakalaması sıfır olabilir. J48'in Python ağacından farklı olması hata kanıtı değildir. Aşağıdaki kontrol WEKA kural metnindeki sayımları Python'un aynı tablosunda yeniden hesaplar; farklı uzunluk veya negatif hedef kuralları dışlanır.

In [ ]:
import re
metin=(cikti/"weka_Apriori.txt").read_text()
kontrol=[]
weka_kume=set()
for satir in metin.splitlines():
    if "==>" not in satir or "hedef=yes" not in satir: continue
    sol,sag=satir.split("==>",1)
    eslesmeler=re.findall(r"([a-z_]+)=([^ ]+)",sol)
    if not 1<=len(eslesmeler)<=2: continue
    maske=np.ones(len(islem),dtype=bool)
    for sutun,deger in eslesmeler:
        maske &= islem[sutun].astype(str).eq(deger).to_numpy()
    ortak_weka=re.search(r"hedef=yes\s+(\d+)",sag)
    oncul_weka=re.search(r"\s(\d+)\s*$",sol)
    assert ortak_weka and oncul_weka, "WEKA sayim bicimi degisti"
    anahtar=frozenset(sutun+"="+deger for sutun,deger in eslesmeler)
    weka_kume.add(anahtar)
    ortak=int((maske & islem.hedef.eq("yes").to_numpy()).sum())
    kontrol.append({"kural":" & ".join(sorted(anahtar)),"python_oncul":int(maske.sum()),
        "weka_oncul":int(oncul_weka.group(1)),"python_ortak":ortak,"weka_ortak":int(ortak_weka.group(1))})
kontrol=pd.DataFrame(kontrol)
python_kume=set(kurallar.antecedents)
assert len(kontrol)>0, "WEKA pozitif kural cikarmadi"
assert (kontrol.python_ortak==kontrol.weka_ortak).all()
assert (kontrol.python_oncul==kontrol.weka_oncul).all()
assert weka_kume==python_kume, {"yalniz_weka":weka_kume-python_kume,"yalniz_python":python_kume-weka_kume}
print("Tum oncul kumeleri ve oncul/ortak sayimlari eslesti:",len(weka_kume))
display(kontrol); kontrol.to_csv(cikti/"weka_kural_kontrol.csv",index=False)
# Derinlik ve agirlik ayri faktorler; tum modeller ayni 10 katta.
weka_katlar=list(StratifiedKFold(10,shuffle=True,random_state=TOHUM).split(egitim,sonuc))
kiyas_modelleri={"Baseline":modeller["Baseline"]}
for derinlik in [6,None]:
    for agirlik in [None,"balanced"]:
        kiyas_modelleri[f"CART derinlik={derinlik}, agirlik={agirlik}"]=hat(DecisionTreeClassifier(
            max_depth=derinlik,min_samples_leaf=30,class_weight=agirlik,random_state=TOHUM))
kiyas_satirlari=[]
for ad,kiyas_modeli in kiyas_modelleri.items():
    kontrol_skor=cross_validate(kiyas_modeli,egitim,sonuc,cv=weka_katlar,
        scoring={**olcutler,"Dogruluk":"accuracy"},n_jobs=1,error_score="raise")
    for anahtar,deger in kontrol_skor.items():
        if anahtar.startswith("test_"):
            kiyas_satirlari.append({"model":ad,"olcut":anahtar[5:],"ortalama":deger.mean(),"std":deger.std(ddof=1)})
cart_kiyas=pd.DataFrame(kiyas_satirlari)
display(cart_kiyas.pivot(index="model",columns="olcut",values=["ortalama","std"]))
cart_kiyas.to_csv(cikti/"kontrollu_cart.csv",index=False)
puanlar.to_csv(cikti/"tum_cv_skorlari.csv",index=False)
protokol={"tohum":TOHUM,"gelistirme_sinir":sinir,"cv":"5 kat x 2 tekrar; katmanlı", "olcut":"average_precision", "secilen":en_iyi_ad,"ileri_test_mcnemar_p":p_degeri,"sinirlar":sinirlar,"surumler":{"python":sys.version,"sklearn":sklearn.__version__,"pandas":pd.__version__,"numpy":np.__version__,"scipy":scipy.__version__}}
(cikti/"protokol.json").write_text(json.dumps(protokol,ensure_ascii=False,indent=2))

protokol["veri_sha256"]=hashlib.sha256(veri_yolu.read_bytes()).hexdigest()
protokol["mcnemar"]={"modeller":ilk_iki,"b":b,"c":c,"p":p_degeri,"esik":0.5}
(cikti/"protokol.json").write_text(json.dumps(protokol,ensure_ascii=False,indent=2))


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Tum oncul kumeleri ve oncul/ortak sayimlari eslesti: 12


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,kural,python_oncul,weka_oncul,python_ortak,weka_ortak
0,ay=mar & kampanya=dilim_1,210,210,107,107
1,ay=mar & iletisim=cellular,238,238,118,118
2,ay=mar & bireysel_kredi=no,241,241,119,119
3,ay=mar,258,258,125,125
4,ay=mar & konut_kredisi=no,190,190,92,92
5,ay=mar & onceki_sonuc=temas_yok,234,234,113,113
6,ay=mar & onceki_sayi=dilim_1,234,234,113,113
7,ay=apr & konut_kredisi=no,701,701,274,274
8,onceki_sonuc=success,210,210,74,74
9,onceki_sayi=dilim_2 & onceki_sonuc=success,210,210,74,74


Görüntülenen çıkış son 5000 satıra kısaltıldı.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tz

ortalama                             \
olcut                                       AP Dengeli_dogruluk  Dogruluk   
model                                                                       
Baseline                              0.067269         0.500000  0.932731   
CART derinlik=6, agirlik=None         0.187629         0.528373  0.933007   
CART derinlik=6, agirlik=balanced     0.195543         0.617294  0.723570   
CART derinlik=None, agirlik=None      0.213085         0.539413  0.931901   
CART derinlik=None, agirlik=balanced  0.206603         0.612418  0.685025   

                                                                               \
olcut                                Duyarlilik        F1  Kesinlik   ROC_AUC   
model                                                                           
Baseline                               0.000000  0.000000  0.000000  0.500000   
CART derinlik=6, agirlik=None          0.060836  0.108130  0.526833  0.654540   
CART derinlik=6, agirlik=balanced      0.494488  0.197972  0.127158  0.671064   
CART derinlik=None, agirlik=None       0.085910  0.144597  0.467644  0.653948   
CART derinlik=None, agirlik=balanced   0.528527  0.184191  0.111545  0.654276   

                                           std                             \
olcut                                       AP Dengeli_dogruluk  Dogruluk   
model                                                                       
Baseline                              0.000131         0.000000  0.000131   
CART derinlik=6, agirlik=None         0.013174         0.008018  0.000974   
CART derinlik=6, agirlik=balanced     0.016053         0.013686  0.069300   
CART derinlik=None, agirlik=None      0.019278         0.009920  0.001934   
CART derinlik=None, agirlik=balanced  0.017221         0.019126  0.009458   

                                                                               
olcut                                Duyarlilik        F1  Kesinlik   ROC_AUC  
model                                                                          
Baseline                               0.000000  0.000000  0.000000  0.000000  
CART derinlik=6, agirlik=None          0.017219  0.027695  0.075969  0.014620  
CART derinlik=6, agirlik=balanced      0.081884  0.020469  0.025246  0.016497  
CART derinlik=None, agirlik=None       0.020044  0.030687  0.074274  0.017093  
CART derinlik=None, agirlik=balanced   0.037587  0.012500  0.007603  0.023030

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

924

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Bağımsız kural kontrolü ve karar:** Çıktı 12 kısa pozitif kuralın öncül kümelerinin tamamının ve tüm öncül/ortak sayımlarının eşleştiğini bildiriyor. Örneğin Mart & kampanya=dilim_1 için her iki araç 210 öncül ve 107 ortak; önceki sonuç=success için 210 öncül ve 74 ortak kayıt buluyor. Bu, belirtilen eşikler ve kısa pozitif kural kapsamı için güçlü bir hesaplama tutarlılığı kontrolüdür; bütün uzunluklar, negatif hedefler veya farklı eşikler için eşitlik iddiası değildir. WEKA daha uzun öğe kümeleri ürettiğinden toplam örüntü sayısı doğrudan Python'un max_len=3 sayısıyla karşılaştırılmaz.

**Ağırlık ve derinliğin ayrıştırılması:** Python'un kendi ortak 10 katında derinlik=6 için ağırlıksız CART duyarlılığı 0,060836 ± 0,017219; balanced sürüm 0,494488 ± 0,081884'tür. Buna karşılık kesinlik 0,526833'ten 0,127158'e, doğruluk 0,933007'den 0,723570'e iner. Daha çok pozitif yakalamak daha çok yanlış pozitif pahasına gerçekleşmiştir. Sınırsız derinlikte aynı ağırlık değişimi duyarlılığı 0,085910'dan 0,528527'ye taşır; dolayısıyla fark yalnız derinlikle açıklanamaz.

Ağırlıksız sınırsız CART AP=0,213085 ± 0,019278; balanced sınırsız CART AP=0,206603 ± 0,017221'dir. Balanced seçeneği her ölçütte üstün değildir. J48'in çok düşük pozitif yakalamasını yalnız sınıf ağırlığı açıklamaz: ağırlıksız CART da J48'den farklıdır. C4.5/CART ayrımı, budama, eksik değer işleme ve farklı kat üyelikleri kalır. Python'da bu dört CART aynı katlarda karşılaştırılır; WEKA'nın seed=42 olması aynı satır bölmelerini garanti etmez. Bu nedenle araçlar arası nedensel bileşen ayrıştırması tamamlanmış gibi sunulmaz.

**Yorum:** Sınıf ağırlığının etkisi aynı derinlikteki CART çiftlerinden okunmalıdır. Derinlik ve ağırlığın aynı anda değiştiği iki satır bu etkiyi ayrıştıramaz. J48 ile kalan farkı tek başına budamaya bağlamak doğru değildir. Kural tamlığı yalnız hedef=yes, en çok iki öncül ve belirtilen eşiklerle tanımlanan kapsamda denetlenmiştir.

In [ ]:
# Ozetin sayilari sabit metinden degil bu calistirmanin nesnelerinden gelir.
display(Markdown("## Bu çalıştırmanın sonuç özeti"))
display(model_puanlari.groupby("deney").AP.agg(["mean","std"]))
display(Markdown(f"Geliştirme pozitif oranı **%{100*sonuc.mean():.2f}**, ileri dönem **%{100*gelecek_sonuc.mean():.2f}**. AP dönemler arasında prevalanstan bağımsız değildir."))
display(Markdown(f"GridSearch best_score_: **{ic_iyi:.4f}**; nested dış AP: **{dis_skor.mean():.4f} ± {dis_skor.std(ddof=1):.4f}**. Farkın pozitif olması zorunlu değildir."))
display(ileri_ozet)
display(Markdown(f"McNemar: **{ilk_iki}**, b={b}, c={c}, exact p={p_degeri:.3g}. Hata oranı farkı (birinci−ikinci): **{(b-c)/len(gelecek_sonuc):.4f}**. Bu AP farkının testi değildir."))
display(Markdown(f"WEKA/Python: **{len(weka_kume)}** kısa pozitif kuralın tüm öncül kümeleri, öncül ve ortak sayımları eşleşti. Model öneminde ilk beş: **{', '.join(onemler.head(5).index)}**."))
print("Sizinti ve kodlama duyarliligi")
display(puanlar.loc[~puanlar.deney.isin(modeller)].groupby("deney").AP.agg(["mean","std"]))


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Bu çalıştırmanın sonuç özeti

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,mean,std
deney,,
Agac,0.193834,0.012401
Artirma,0.251932,0.015005
Baseline,0.067269,0.000067
Lojistik,0.213244,0.010024
Orman,0.230550,0.010659
kNN,0.157351,0.007657


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Geliştirme pozitif oranı **%6.73**, ileri dönem **%31.58**. AP dönemler arasında prevalanstan bağımsız değildir.

... (9 outputs left out)

Sizinti ve kodlama duyarliligi


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,mean,std
deney,,
gosterge_False,0.213178,0.009935
gosterge_True,0.213244,0.010024
kodlama_birebir,0.213244,0.010024
kodlama_hedef,0.213406,0.010623
kodlama_nadir,0.209605,0.008928
sizinti_dogru,0.213244,0.010024
sizinti_hedef_yanlis,0.213357,0.010997
sizinti_onislem_yanlis,0.213278,0.009864


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Son değerlendirme ve yeniden çalıştırma sınırı:** Dinamik sonuç özeti, korunan kayıtlı tablolarla tutarlıdır: CV'ye göre Artırma seçilir; ilk iki model Artırma/Orman'dır; McNemar b=1.359, c=1.611; eşleşen kısa pozitif kural sayısı 12; önem sırası önceki sonuç, ay, yaş, gün, bakiyedir. Önceki sohbet veya farklı sürümlerdeki orman/boosting değerleri bu defterin çıktıları yerine kullanılmamıştır. Bu yorum revizyonunda kod, değişken adları, çalıştırma sayaçları ve kayıtlı sonuçlar değiştirilmemiştir. Yeni baştan sona çalıştırma yapılmamıştır; yeniden çalıştırmada değişen dinamik çıktılar esas alınmalı ve burada kayıtlı çalıştırmaya bağlanan sayısal açıklamalar tekrar gözden geçirilmelidir.

Ana sonuç, doğru kurulumda AP bakımından seçilen modelin her eşik ölçütünde en iyi olmadığı; eksiklik ve dönem değişiminin yorum için belirleyici olduğudur. Veri küpünde seçilen yaş katmanları Simpson tersine dönüşü göstermemiş, Apriori ise küçük ve kampanya dönemine bağlı profiller üretmiştir. Bu bulgular gözlemseldir ve kampanya müdahalesinin etkisini ölçmez. İleri dönem daha önce görülmüş ve açıklama analizlerinde kullanılmıştır; gelecekteki araştırma kararları için yeni bağımsız dönem gerekir.

**Yorum:** Bu özet her çalıştırmada güncellenir. Ana seçim CV AP’ye dayanır; ileri dönem sonuçları görüldükten sonraki düzeltmeler yeni bağımsız doğrulama değildir. Colab web oturumunda ayrıca baştan sona çalıştırılmalıdır.

**Güven aralığı ekinin doğrulama notu:** Bu ekte veri hazırlama, veri küpü ve Apriori yeniden hesaplanarak yeni aralık hücreleri çalıştırılmıştır. Orijinal model eğitimleri, CV ve WEKA yeniden çalıştırılmamış; bunların kodları ve kayıtlı çıktıları korunmuştur. Yeni hücrelerin çıktıları gerçek sayımlardan gelir. Önceki yorum revizyonundaki “yeni analiz yapılmadı” ifadesi yalnız o revizyonu anlatır; bu ek sürüm iki yeni analiz hücresi içerir.